In [1]:
import os
# python standard library imports
from pathlib import Path
import json
import math
# model building imports
import tensorflow as tf
from keras import Model, layers, Sequential
from keras.applications import EfficientNetV2S, Xception, xception
# model training imports
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
# other imports
from keras.utils import image_dataset_from_directory

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_addons as tfa

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")


# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
tf.config.optimizer.set_jit(True)
print("XLA JIT enabled.")

GPU detected: ['/physical_device:GPU:0']
XLA JIT enabled.


## Model definition

In [3]:
class ResidualBlock(layers.Layer):
    """
    Single residual block: Conv → BN → Activation + shortcut projection.

    Storing conv/bn/activation as named attributes of a Layer subclass
    guarantees Keras tracks their weights correctly.
    The original bug stored these inside plain Python dicts inside a plain
    Python list — Keras never registered them, so they were never trained.
    """

    def __init__(self, filters, kernel_size, stride, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.filters     = filters
        self.kernel_size = kernel_size
        self.stride      = stride
        self.activation  = activation

        self.conv     = layers.Conv2D(filters, kernel_size, strides=stride,
                                      padding="same")
        self.bn       = layers.BatchNormalization()
        self.actv     = layers.Activation(activation)
        self.shortcut = layers.Conv2D(filters, (1, 1), strides=stride,
                                      padding="same")
        self.add      = layers.Add()

    def call(self, x, training=False):
        skip = self.shortcut(x)
        x    = self.conv(x)
        x    = self.bn(x, training=training)
        x    = self.actv(x)
        # Single activation AFTER the residual add — standard ResNet convention
        return self.add([x, skip])

    def get_config(self):
        return {**super().get_config(),
                "filters": self.filters, "kernel_size": self.kernel_size,
                "stride": self.stride,   "activation": self.activation}

In [4]:
class MyCNN(Model):
    def __init__(self, conv_configs, dense_configs, num_classes, augmentation_layer=None, activation="relu", dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="my_cnn")
        self.num_classes = num_classes
        self.conv_configs = conv_configs
        self.dense_configs = dense_configs
        self.augmentation_layer = augmentation_layer
        self.activation = activation
        self.dropout_rate = dropout_rate

        # 1. ADD RESCALING HERE (The fix for your Transfer Learning compatibility)
        self.rescaling = layers.Rescaling(1./255)

        # Store as a Python list of Layer objects assigned to self.
        # Keras DOES track a list of Layers set as an attribute via __setattr__,
        # as long as the list itself is set at attribute assignment time (not grown later).
        # Safest pattern: build the full list first, then assign once.
        self.blocks = [
            ResidualBlock(f, k, s, activation=activation,
                          name=f"block_{i}")
            for i, (f, k, s) in enumerate(conv_configs)
        ]

        self.gap = layers.GlobalAveragePooling2D(name="GAP")
        dense_list = []
        for i, u in enumerate(self.dense_configs):
            dense_list.append(layers.Dense(u, activation=self.activation, name=f"fc_{i}"))
            dense_list.append(layers.Dropout(self.dropout_rate, name=f"drop_{i}"))
        self.dense_layers = dense_list
        self.classifier = layers.Dense(self.num_classes, activation='softmax', name="head")

    def get_config(self):
        # Obtém a configuração base da superclasse
        config = super().get_config()
        # Adiciona os teus argumentos personalizados ao dicionário
        config.update({
            "num_classes": self.num_classes,
            "conv_configs": self.conv_configs,
            "dense_configs": self.dense_configs,
            "augmentation_layer": self.augmentation_layer,
            "activation": self.activation,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        x = self.rescaling(inputs)
        if self.augmentation_layer is not None:
            x = self.augmentation_layer(x, training=training)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.gap(x)
        for layer in self.dense_layers:
            # Dropout needs training flag; Dense does not
            x = layer(x, training=training) if isinstance(layer, layers.Dropout) else layer(x)
        return self.classifier(x)

## Config and data loading

In [5]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
IMAGE_SIZE     = (384, 384)
BATCH_SIZE     = 16       # reduced from 32 to fit 384px images in 8GB VRAM
EPOCHS         = 64       # good balance
LEARNING_RATE  = 1e-1     # 2x LR for every 2x in batch size
N_CLASSES      = 23

data_dir_path = Path("..\wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


## Augmentation and Mixup

In [6]:
# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
# Forces the model to learn smoother decision boundaries rather than
# memorising exact compositions — especially useful for fine-grained style tasks.
def mixup(images, labels, alpha=0.4):
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

train_ds_mixed = (
    train_ds
    .map(mixup, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)
# val and test are never augmented or mixed
val_ds  = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

# ── Custom CNN architecture config ───────────────────────────────────────────
conv_setup = [
    (64,  (7, 7), 2),
    (64,  (3, 3), 1),
    (128, (3, 3), 2),
    (128, (3, 3), 1),
    (256, (3, 3), 2),
    (256, (3, 3), 1),
    (512, (3, 3), 2),
    (512, (3, 3), 1),
]
dense_setup = [1024, 512]

# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}


## Model instantiation

In [7]:
model = MyCNN(
    augmentation_layer=None,
    conv_configs=conv_setup,
    dense_configs=dense_setup,
    num_classes=N_CLASSES,
)

## Metrics and loss

In [8]:
def make_metrics(num_classes):
    """Fresh metric instances per model — metrics are stateful and must not be shared."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        tfa.metrics.F1Score(num_classes=num_classes, average="macro", name="f1_score")
    ]

## Learning rate schedule — cosine annealing with warmup

In [9]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Linear warmup then cosine annealing.

    Warmup matters especially for the larger LR used with MyCNN (2e-3):
    without it, the first few batches produce outsized gradient updates.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler


## Compile and prepare callbacks

In [10]:
# Compile the model
model.compile(
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1), 
    optimizer=SGD(learning_rate=LEARNING_RATE, name="optimizer", decay=0.01), 
    metrics=make_metrics(num_classes=N_CLASSES)
)

In [11]:
# Define Callbacks
checkpoint_callback = ModelCheckpoint(
    checkpoints_folder_path / f"checkpoint_{model.name}",
    save_best_only=True,
    monitor="val_loss",
    verbose=0
)
metrics_callback = CSVLogger(metrics_folder_path / f"metric_{model.name}.csv")

In [12]:
lr_scheduler_callback = LearningRateScheduler(make_cosine_warmup_scheduler(LEARNING_RATE, EPOCHS, warmup_epochs=3))

In [13]:
# EarlyStopping: stops training if val_loss doesn't improve for 7 epochs
# and restores the best weights automatically
early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)


In [14]:
callbacks = [
    checkpoint_callback,
    metrics_callback,
    lr_scheduler_callback,
    early_stopping_callback
]

## Train the model

In [15]:
# Train the model
model_fit_data = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)
model_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
# Limpa a memória da GPU/RAM ocupada pelo modelo que acabou de treinar
clear_session()

model_fit_data, model_eval_data

Epoch 1/64
583/583 [==============================] - ETA: 0s - loss: 3.1268 - accuracy: 0.1177 - auc: 0.5655 - f1_score: 0.0853

INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


583/583 [==============================] - 173s 257ms/step - loss: 3.1268 - accuracy: 0.1177 - auc: 0.5655 - f1_score: 0.0853 - val_loss: 2.8697 - val_accuracy: 0.1943 - val_auc: 0.7475 - val_f1_score: 0.1371 - lr: 0.0333
Epoch 2/64
583/583 [==============================] - ETA: 0s - loss: 2.9712 - accuracy: 0.1713 - auc: 0.6155 - f1_score: 0.1273

INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


...

583/583 [==============================] - 219s 375ms/step - loss: 2.5850 - accuracy: 0.3283 - auc: 0.7021 - f1_score: 0.2690 - val_loss: 2.3302 - val_accuracy: 0.3614 - val_auc: 0.8712 - val_f1_score: 0.3401 - lr: 0.0092
Epoch 54/64
583/583 [==============================] - 211s 362ms/step - loss: 2.5888 - accuracy: 0.3321 - auc: 0.7025 - f1_score: 0.2710 - val_loss: 2.3313 - val_accuracy: 0.3609 - val_auc: 0.8713 - val_f1_score: 0.3390 - lr: 0.0078
Epoch 55/64
583/583 [==============================] - 217s 372ms/step - loss: 2.5847 - accuracy: 0.3293 - auc: 0.7016 - f1_score: 0.2693 - val_loss: 2.3320 - val_accuracy: 0.3614 - val_auc: 0.8712 - val_f1_score: 0.3392 - lr: 0.0065
Epoch 56/64
583/583 [==============================] - 247s 424ms/step - loss: 2.5837 - accuracy: 0.3300 - auc: 0.7029 - f1_score: 0.2712 - val_loss: 2.3317 - val_accuracy: 0.3589 - val_auc: 0.8712 - val_f1_score: 0.3373 - lr: 0.0053
Epoch 57/64
583/583 [==============================] - ETA: 0s - loss: 2.583

INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


INFO:tensorflow:Assets written to: Checkpoints\checkpoint_my_cnn\assets


583/583 [==============================] - 220s 377ms/step - loss: 2.5838 - accuracy: 0.3277 - auc: 0.7017 - f1_score: 0.2663 - val_loss: 2.3296 - val_accuracy: 0.3594 - val_auc: 0.8715 - val_f1_score: 0.3382 - lr: 0.0042
Epoch 58/64
583/583 [==============================] - 377s 647ms/step - loss: 2.5859 - accuracy: 0.3320 - auc: 0.7018 - f1_score: 0.2719 - val_loss: 2.3297 - val_accuracy: 0.3599 - val_auc: 0.8716 - val_f1_score: 0.3383 - lr: 0.0032
Epoch 59/64
583/583 [==============================] - 189s 325ms/step - loss: 2.5847 - accuracy: 0.3294 - auc: 0.7008 - f1_score: 0.2691 - val_loss: 2.3311 - val_accuracy: 0.3574 - val_auc: 0.8709 - val_f1_score: 0.3364 - lr: 0.0024
Epoch 60/64
583/583 [==============================] - 227s 390ms/step - loss: 2.5846 - accuracy: 0.3300 - auc: 0.7029 - f1_score: 0.2713 - val_loss: 2.3305 - val_accuracy: 0.3589 - val_auc: 0.8713 - val_f1_score: 0.3376 - lr: 0.0016
Epoch 61/64
583/583 [==============================] - 222s 381ms/step - los

(<keras.callbacks.History at 0x2330b159750>,
 {'loss': 2.3383331298828125,
  'accuracy': 0.34915924072265625,
  'auc': 0.8718570470809937,
  'f1_score': 0.32780909538269043})